In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import matplotlib.pyplot as plt
import seaborn as sns

from itertools import product
import joblib
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm
import os, sys, logging, gc

sys.path.append('../')
import src.forecasting.simulations      as sim
import src.fda.kde.estimators           as kde
import src.fda.transformations.lqdt     as lqdt
import src.forecasting.cross_validation as cv
import src.forecasting.pipelines        as fp
import src.forecasting.accuracy         as acc
import src.fda.utils                    as fdaUtils

In [3]:
EXECUTION_DATE : str = datetime.now().strftime('%Y%m%d')
EXECUTION_DATE = '20260604_02'

LOG_PATH        : str = f'../logs/simulations/{EXECUTION_DATE}.log'
SIM_MAIN_PATH   : str = f'../data/interim/simulation/{EXECUTION_DATE}/'
SIMS_PATH       : str = f'../data/interim/simulation/{EXECUTION_DATE}/00_simulations/'
SIMS_LQD_PATH   : str = f'../data/interim/simulation/{EXECUTION_DATE}/01_lqds/'
KDES_PATH       : str = f'../data/interim/simulation/{EXECUTION_DATE}/02_kdes/'
KDES_LQDS_PATH  : str = f'../data/interim/simulation/{EXECUTION_DATE}/03_kde_lqds/'
CV_FCS_PATH     : str = f'../data/interim/simulation/{EXECUTION_DATE}/04_cv/'

Path(SIM_MAIN_PATH).mkdir(parents=True, exist_ok=True)
Path(SIMS_PATH).mkdir(parents=True, exist_ok=True)
Path(SIMS_LQD_PATH).mkdir(parents=True, exist_ok=True)
Path(KDES_PATH).mkdir(parents=True, exist_ok=True)
Path(KDES_LQDS_PATH).mkdir(parents=True, exist_ok=True)
Path(CV_FCS_PATH).mkdir(parents=True, exist_ok=True)
Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

# GAS model

In [4]:
# Define scenarios
gas_params = {
    "scenario_1": {
        "Description": "Location-driven ($m_t$)",
        "alpha": np.diag([0.08, 0.01, 0.01]),
        "beta": np.diag([0.90, 0.95, 0.95]),
    },
    "scenario_2": {
        "Description": "Scale-driven ($\\sigma_t$)",
        "alpha": np.diag([0.01, 0.08, 0.01]),
        "beta": np.diag([0.95, 0.90, 0.95]),
    },
    "scenario_3": {
        "Description": "Shape-driven ($\\eta_t$)",
         "alpha": np.diag([0.01, 0.01, 0.08]),
          "beta": np.diag([0.95, 0.95, 0.90])
    },
    "scenario_4": {
        "Description": "Mixture",
        "alpha": np.diag([0.04, 0.06, 0.04]),
        "beta": np.diag([0.92, 0.95, 0.94]),
    }
}

distribution_params = {
    "nu": [3, 8]
}

In [5]:
keys = ["scenario", "nu"]
values = [list(gas_params.keys()), distribution_params["nu"]]

param_grid = []

for scenario, nu in product(*values):
    gas_cfg = gas_params[scenario]

    param_grid.append({
        "scenario": "___nu=".join([scenario, str(nu)]),
        "nu": nu,
        "alpha": gas_cfg["alpha"],
        "beta": gas_cfg["beta"]    
})

In [6]:
# grid for densities
x = np.linspace(-40, 40, 5001)
# number of curves (densities)
T = 301
# number of simulations (f_{N_REP,1},...,f_{N_REP,T})
N_REPS = 1_0
# number of samples from each f_t density
N_SAMPLES = 288

HORIZON = 1
CV_WINDOW_TYPE = "expanding"
CV_WINDOW_SIZE = T - HORIZON

SIMULATION_END_DATE = pd.Timestamp.today().normalize()
total = len(param_grid) * N_REPS

In [7]:
total = len(param_grid) * N_REPS
pbar = tqdm(total=total, desc="Total Simulation Progress")

sim_addresses = []

for params in param_grid:
    scenario = params["scenario"]

    for n_rep in range(N_REPS):
        sim_path = f"{SIMS_PATH}{scenario}___rep_{n_rep}.jbl"

        if Path(sim_path).exists():
            sim_addresses.append({
                "scenario": scenario,
                "n_rep": n_rep,
                "address": sim_path
            })
            pbar.update(1)
            continue

        # 1. Setup Model and Simulate
        gm = sim.GasModel(alpha=params["alpha"], beta=params["beta"], nu=params["nu"])
        sim_results = gm.simulate(T=T, burn_in=300)

        # 2. Get Theoretical Conditional Densities
        sim_density = gm.conditional_densities(grid=x, theta_path=sim_results["theta"])
        dates = pd.date_range(end=SIMULATION_END_DATE, periods=T, freq="D")
        sim_density.columns = dates

        # 3. Generate Samples Efficiently
        samples_list = []
        for i in range(len(sim_results["theta"])):
            sample = gm.rvs(n=N_SAMPLES, theta=sim_results["theta"][i])
            samples_list.append(sample)

        df_samples = pd.DataFrame(np.array(samples_list).T, columns=dates)

        rep_result = {
            "scenario": scenario,
            "n_rep": n_rep,
            "params": params,
            "theta": sim_results["theta"],
            "densities": sim_density,
            "samples": df_samples
        }

        joblib.dump(rep_result, sim_path)
        sim_addresses.append({
            "scenario": scenario,
            "n_rep": n_rep,
            "address": sim_path
        })

        del gm, sim_results, sim_density, samples_list, df_samples, rep_result
        gc.collect()
        pbar.update(1)

joblib.dump(sim_addresses, f"{SIM_MAIN_PATH}simulation_addresses.jbl")

Total Simulation Progress:   0%|          | 0/80 [00:00<?, ?it/s]

['../data/interim/simulation/20260604_02/simulation_addresses.jbl']

In [8]:
sim_jbls_path = SIMS_PATH
sim_paths = [
    str(Path(sim_jbls_path) / x)
    for x in os.listdir(sim_jbls_path)
    if x.endswith(".jbl") and "___rep_" in x
]

sim_addresses = []
for path_name in sim_paths:
    obj = joblib.load(path_name)
    sim_address = {
        "scenario": obj["scenario"],
        "n_rep": obj["n_rep"],
        "address": path_name
    }
    sim_addresses.append(sim_address)
    del obj

sim_addresses = sorted(
    sim_addresses,
    key=lambda x: (x["scenario"], x["n_rep"])
)

sim_index = {
    (d["scenario"], d["n_rep"]): d["address"]
    for d in sim_addresses
}

joblib.dump(sim_addresses, f"{SIM_MAIN_PATH}simulation_addresses.jbl")
print(f"Indexed {len(sim_addresses)} simulation files")

Indexed 80 simulation files


In [ ]:
# # Plotting simulations
# doc_path = "../../densities4risk_doc/Figures/"
# # ==========================================
# # Metadata
# # ==========================================

# nus = distribution_params["nu"]
# scenario_names = list(gas_params.keys())

# # ==========================================
# # Figure
# # ==========================================

# n_rows = len(scenario_names)
# n_cols = len(nus)

# fig, axes = plt.subplots(
#     n_rows,
#     n_cols,
#     figsize=(5 * n_cols, 3.5 * n_rows),
#     dpi=300,
#     constrained_layout=True,
#     squeeze=False
# )

# # ==========================================
# # Plot loop
# # ==========================================

# for i, scenario in enumerate(scenario_names):

#     for j, nu in enumerate(nus):

#         ax = axes[i, j]

#         key = f"{scenario}___nu={nu}"

#         sim_path = sim_index.get((key, 0))
#         if sim_path is None:
#             ax.set_visible(False)
#             continue

#         sim_obj = joblib.load(sim_path)
#         data = sim_obj["densities"]

#         # ----------------------------------
#         # Background densities
#         # ----------------------------------

#         data.plot(
#             color="gray",
#             alpha=0.03,
#             linewidth=0.5,
#             legend=False,
#             ax=ax
#         )

#         ax.set_xlim([-10,10])

#         # ----------------------------------
#         # First curve
#         # ----------------------------------

#         first_line = data.iloc[:, 0].plot(
#             color="#70e000",
#             linewidth=2.2,
#             ax=ax,
#             label="t=1"
#         )

#         # ----------------------------------
#         # Last curve
#         # ----------------------------------

#         last_line = data.iloc[:, -1].plot(
#             color="#e5383b",
#             linewidth=2.2,
#             ax=ax,
#             label="t=T"
#         )

#         # ==================================
#         # Titles / labels
#         # ==================================

#         if i == 0:
#             ax.set_title(
#                 rf"$\nu={nu}$",
#                 fontsize=15,
#                 fontweight="bold",
#                 pad=10
#             )

#         if j == 0:
#             ax.set_ylabel(
#                 gas_params[scenario]["Description"],
#                 # scenario.replace("_", " ").title() + "\nDensity",
#                 fontsize=12
#             )

#         ax.set_xlabel(r"$x$", fontsize=11)

#         # ==================================
#         # Styling
#         # ==================================

#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)

#         ax.grid(
#             True,
#             linestyle="--",
#             alpha=0.2
#         )

# # ==========================================
# # Shared legend
# # ==========================================

# handles = [
#     axes[0, 0].lines[-2],
#     axes[0, 0].lines[-1]
# ]

# labels = ["t=1", "t=T"]

# fig.legend(
#     handles,
#     labels,
#     loc="upper center",
#     ncol=2,
#     frameon=False,
#     fontsize=11
# )

# # plt.savefig(''.join([doc_path, "sinape_gasSim_all.png"]), bbox_inches='tight') # PDFs are better for LaTeX
# plt.show()

# KDE

In [9]:
rot_grid = {
    "method": ["rot"], #rot/2, --> h elevado reduz as dinâmicas captadas pelo estimador (não considerar 2*ROT)
    "kernel": ["gaussian"],
    "sigma_robust": [False]
}

density_param_grid = {}
for grid in [rot_grid]:
    for params in ParameterGrid(grid):
        
        key_parts = [params["kernel"]]
        if "method" in params: key_parts.append(params["method"])
        if "df" in params: key_parts.append(f"df={params['df']}")
        if params.get("sigma_robust"): key_parts.append("robust")
        if params.get("cv") == "LOO": key_parts.append("loo")
        
        full_model_name = "_".join(key_parts).replace(".", "")
        
        kernel_label = params["kernel"]
        if "df" in params:
            kernel_label += f"+df={params['df']}"
            
        bw_label = params.get("method", "fixed")
        if params.get("sigma_robust"): bw_label += "_robust"
        if params.get("cv") == "LOO": bw_label += "_loo"


        density_param_grid[full_model_name] = {
            "kernel": kernel_label,
            "bandwidth": bw_label,
            "params": params  
        }

print("KDE models:")
for name, values in density_param_grid.items():
    print("\t",name, ":", values["params"])

print(f"Total KDE models: {len(density_param_grid.items())}")

KDE models:
	 gaussian_rot : {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': False}
Total KDE models: 1


In [10]:
total = len(sim_addresses) * len(density_param_grid)
pbar = tqdm(total=total, desc="Total KDE Progress")

for sim_address in sim_addresses:
    scenario = sim_address["scenario"]
    n_rep = sim_address["n_rep"]
    scenario_sim_dict = joblib.load(sim_address["address"])
    returns_df = scenario_sim_dict["samples"]

    print(f"Processing KDEs for {scenario} | rep {n_rep}")

    for kde_bw_name, kde_bw_params in density_param_grid.items():
        kde_path = f"{KDES_PATH}{scenario}___rep_{n_rep}___kde_{kde_bw_name}.jbl"

        if Path(kde_path).exists():
            pbar.update(1)
            continue

        print(f"	{kde_bw_name}")

        # bandwidths
        df_h = kde.df_bandwidth_selector(returns_df, **kde_bw_params["params"])
        kde_params = {k: v for k, v in kde_bw_params["params"].items() if k in ["kernel", "df"]}

        # KDEs for samples from f_t
        df_grids, df_densities = kde.df_to_kde(
            X=returns_df,
            h=df_h,
            normalize_densities=False,
            **kde_params
        )

        rep_result = {
            "scenario": scenario,
            "n_rep": n_rep,
            "model_name": kde_bw_name,
            "kde_params": kde_bw_params["kernel"],
            "kernel": kde_bw_params["params"]["kernel"],
            "bw_params": kde_bw_params["bandwidth"],
            "bw_method": kde_bw_params["params"]["method"],
            "df_h": df_h,
            "df_support": df_grids,
            "df_densities": df_densities
        }

        joblib.dump(rep_result, kde_path)

        del df_h, df_grids, df_densities, kde_params, rep_result
        gc.collect()
        pbar.update(1)

    del scenario_sim_dict, returns_df
    gc.collect()

Total KDE Progress:   0%|          | 0/80 [00:00<?, ?it/s]

Processing KDEs for scenario_1___nu=3 | rep 0
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 1
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 2
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 3
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 4
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 5
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 6
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 7
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 8
	gaussian_rot
Processing KDEs for scenario_1___nu=3 | rep 9
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | rep 0
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | rep 1
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | rep 2
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | rep 3
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | rep 4
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | rep 5
	gaussian_rot
Processing KDEs for scenario_1___nu=8 | 

In [11]:
kde_jbls_path = KDES_PATH
kdes_paths = [
    str(Path(kde_jbls_path) / x)
    for x in os.listdir(kde_jbls_path)
    if x.endswith(".jbl") and "___rep_" in x
]

kde_addresses = []
for path_name in kdes_paths:
    obj = joblib.load(path_name)
    kde_address = {
        "scenario":   obj["scenario"],
        "n_rep":      obj["n_rep"],
        "model_name": obj["model_name"],
        "address":    path_name
    }
    kde_addresses.append(kde_address)

kde_addresses = sorted(
    kde_addresses,
    key=lambda x: (x["scenario"], x["n_rep"], x["model_name"])
)

kde_index = {
    (d["scenario"], d["n_rep"], d["model_name"]): d["address"]
    for d in kde_addresses
}

joblib.dump(kde_addresses, f"{SIM_MAIN_PATH}kde_addresses.jbl")
print(f"Indexed {len(kde_addresses)} KDE files")

Indexed 80 KDE files


# LQD transforms

In [12]:
n_sim_addresses = len(sim_addresses)
pbar = tqdm(total=n_sim_addresses, desc="Total LQD(f conditional) Progress")

for sim_address in sim_addresses:
    scenario = sim_address["scenario"]
    n_rep = sim_address["n_rep"]
    sim_lqd_path = f"{SIMS_LQD_PATH}lqd___scenario_{scenario}___rep_{n_rep}.jbl"

    if Path(sim_lqd_path).exists():
        pbar.update(1)
        continue

    scenario_sim_dict = joblib.load(sim_address["address"])
    sim_densities = scenario_sim_dict["densities"]
    sim_densities_supp = sim_densities.copy()
    sim_densities_supp.loc[:, :] = sim_densities_supp.index.to_numpy()[:, None]

    mlqdt = lqdt.mLQDT()
    model_lqd = mlqdt.transform(
        densities=sim_densities,
        densities_supports=sim_densities_supp,
        verbose=False
    )

    conditional_mlqdt = model_lqd.lqd.copy()
    conditional_mlqdt.index = model_lqd.lqd_support

    sim_mlqdt_result = {
        "scenario": scenario,
        "n_rep": n_rep,
        "conditional_mlqdt": conditional_mlqdt,
        "c": model_lqd.c,
        "t0": model_lqd.t0
    }

    joblib.dump(sim_mlqdt_result, sim_lqd_path)

    del scenario_sim_dict, sim_densities, sim_densities_supp, mlqdt, model_lqd, conditional_mlqdt, sim_mlqdt_result
    gc.collect()
    pbar.update(1)


Total LQD(f conditional) Progress:   0%|          | 0/80 [00:00<?, ?it/s]

In [13]:
sim_lqd_paths = [
    str(Path(SIMS_LQD_PATH) / x)
    for x in os.listdir(SIMS_LQD_PATH)
    if x.endswith(".jbl") and "___rep_" in x
]

sim_lqd_addresses = []
for path_name in sim_lqd_paths:
    obj = joblib.load(path_name)
    sim_lqd_address = {
        "scenario": obj["scenario"],
        "n_rep": obj["n_rep"],
        "address": path_name
    }
    sim_lqd_addresses.append(sim_lqd_address)
    del obj

sim_lqd_addresses = sorted(
    sim_lqd_addresses,
    key=lambda x: (x["scenario"], x["n_rep"])
)

sim_lqd_index = {
    (d["scenario"], d["n_rep"]): d["address"]
    for d in sim_lqd_addresses
}

joblib.dump(sim_lqd_addresses, f"{SIM_MAIN_PATH}simulation_lqd_addresses.jbl")
print(f"Indexed {len(sim_lqd_addresses)} conditional LQD files")


Indexed 80 conditional LQD files


In [14]:
n_kde_addresses = len(kde_addresses)
pbar = tqdm(total=n_kde_addresses, desc="Total LQD(fhat) Progress")

for kde_address in kde_addresses:
    scenario_kde_dict = joblib.load(kde_address["address"])
    id_scenario_kde = scenario_kde_dict["scenario"]
    n_rep = scenario_kde_dict["n_rep"]
    model_name_path = scenario_kde_dict["model_name"]

    kde_lqd_path = f"{KDES_LQDS_PATH}lqd___scenario_{id_scenario_kde}___rep_{n_rep}___kde_{model_name_path}.jbl"

    if Path(kde_lqd_path).exists():
        pbar.update(1)
        del scenario_kde_dict
        continue

    mlqdt = lqdt.mLQDT()
    model_lqd = mlqdt.transform(
        densities=scenario_kde_dict["df_densities"],
        densities_supports=scenario_kde_dict["df_support"],
        verbose=False
    )

    Y_t = model_lqd.lqd.copy()
    Y_t.index = model_lqd.lqd_support

    kde_mlqdt_result = {
        "scenario": kde_address["scenario"],
        "n_rep": kde_address["n_rep"],
        "model_name": kde_address["model_name"],
        "kde_mlqdt": Y_t
    }

    joblib.dump(kde_mlqdt_result, kde_lqd_path)

    del scenario_kde_dict, mlqdt, model_lqd, Y_t, kde_mlqdt_result
    gc.collect()
    pbar.update(1)


Total LQD(fhat) Progress:   0%|          | 0/80 [00:00<?, ?it/s]

/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: overflow encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo


In [16]:
kdes_lqd_paths = [
    str(Path(KDES_LQDS_PATH) / x)
    for x in os.listdir(KDES_LQDS_PATH)
    if x.endswith(".jbl") and "___rep_" in x
]

kdes_lqd_addresses = []
for path_name in kdes_lqd_paths:
    obj = joblib.load(path_name)
    kde_address = {
        "scenario": obj["scenario"],
        "n_rep": obj["n_rep"],
        "model_name": obj["model_name"],
        "address": path_name
    }
    kdes_lqd_addresses.append(kde_address)
    del obj

kdes_lqd_addresses = sorted(
    kdes_lqd_addresses,
    key=lambda x: (x["scenario"], x["n_rep"], x["model_name"])
)

kde_lqd_index = {
    (d["scenario"], d["n_rep"], d["model_name"]): d["address"]
    for d in kdes_lqd_addresses
}

joblib.dump(kdes_lqd_addresses, f"{SIM_MAIN_PATH}kde_lqd_addresses.jbl")
print(f"Indexed {len(kdes_lqd_addresses)} KDE LQD files")


Indexed 80 KDE LQD files


# Cross-validation

In [17]:
logger = logging.getLogger("Density_Estimation")
logger.setLevel(logging.DEBUG)

log_file = str(Path(LOG_PATH).resolve())
if not any(isinstance(handler, logging.FileHandler) and handler.baseFilename == log_file for handler in logger.handlers):
    file_handler = logging.FileHandler(LOG_PATH)
    file_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
    logger.addHandler(file_handler)

logging.captureWarnings(True)


In [ ]:
KdFPC_kwargs = {
    "p": 5,
    "dimension": 3
}

In [19]:
def append_curve_comparison(
    measures,
    *,
    scenario,
    n_rep,
    model_name,
    fold,
    comparison,
    fc_date,
    test,
    forecast,
    support=None
):
    acc_measures = acc.overall_measures(forecast=forecast, test=test)
    support_values = test.index if support is None else support

    temp_df = pd.DataFrame({
        "support": support_values,
        "actual": test.iloc[:, 0].values,
        "forecast": forecast.iloc[:, 0].values,
        "date": fc_date,
        "fold": fold
    }).set_index(["fold", "date", "support"])

    measures.append({
        "scenario": scenario,
        "n_rep": n_rep,
        "model_name": model_name,
        "fold": fold,
        "comparison": comparison,
        "fc_date": fc_date,
        **acc_measures,
        "curves": temp_df
    })

In [20]:
total_models = len(kde_addresses)
pbar = tqdm(total=total_models, desc="Total CV Progress")

logger.info("\nInitiating cross validation...")
for kde_address in kde_addresses:
    print(kde_address)
    scenario_kde_dict = joblib.load(kde_address["address"])

    # KDE database
    id_scenario_kde = scenario_kde_dict["scenario"]
    n_rep = scenario_kde_dict["n_rep"]
    model_name_path = scenario_kde_dict["model_name"]
    model_name = scenario_kde_dict["model_name"].replace("_", " ")

    # Simulation database, loaded only for this scenario/replication
    sim_address = sim_index.get((id_scenario_kde, n_rep))
    if sim_address is None:
        raise FileNotFoundError(f"Missing simulation address for {(id_scenario_kde, n_rep)}")

    scenario_sim_dict = joblib.load(sim_address)
    sim_info = scenario_sim_dict["params"]

    logger.info(
        f"Scenario: {id_scenario_kde} | n_rep: {n_rep} | KDE model: {model_name}"
    )

    df_support, df_densities = scenario_kde_dict["df_support"], scenario_kde_dict["df_densities"]

    # Conditional simulated densities f_t
    sim_densities = scenario_sim_dict["densities"]
    sim_densities_supp = sim_densities.copy()
    sim_densities_supp.loc[:, :] = sim_densities_supp.index.to_numpy()[:, None]

    # LQD of conditional densities T(f_t)
    sim_lqd_address = sim_lqd_index.get((id_scenario_kde, n_rep))
    if sim_lqd_address is None:
        raise FileNotFoundError(f"Missing conditional LQD address for {(id_scenario_kde, n_rep)}")
    scenario_sim_lqd = joblib.load(sim_lqd_address)["conditional_mlqdt"]
    # LQD of KDE densities T(fhat_t)
    kde_lqd_address = kde_lqd_index.get((id_scenario_kde, n_rep, scenario_kde_dict["model_name"]))
    if kde_lqd_address is None:
        raise FileNotFoundError(f"Missing KDE LQD address for {(id_scenario_kde, n_rep, scenario_kde_dict['model_name'])}")
    scenario_mlqdt = joblib.load(kde_lqd_address)
    scenario_kde_lqd = scenario_mlqdt["kde_mlqdt"]
    windows = cv.cv_window(T=df_densities.shape[1], h=HORIZON, window_type=CV_WINDOW_TYPE, window_size=CV_WINDOW_SIZE)
    measures = []

    for fold, window in enumerate(windows):
        fold += 1
        print(f"		>>> cv {fold}/{len(windows)}")
        idx_train = window[0]
        idx_test = window[1]
        test_date = df_densities.columns[idx_test]
        fc_date = test_date[0]

        # Targets
        Y_cond_t = scenario_sim_lqd.loc[:, test_date]
        Y_kde_t = scenario_kde_lqd.loc[:, test_date]
        f_hat_t_supp, f_hat_t = df_support.loc[:, test_date], df_densities.loc[:, test_date]
        f_t_support, f_t = sim_densities_supp.loc[:, test_date], sim_densities.loc[:, test_date]

        # Train-test split from KDE densities
        kde_train_support, kde_train = df_support.iloc[:, idx_train], df_densities.iloc[:, idx_train]

        # Forecasting fhat_{t+1}; returns both T-space and inverse-density forecasts
        forecaster = fp.DensityForecaster(kdfpc_kwargs=KdFPC_kwargs.copy(), maxlags=5)
        forecaster.fit(kde_train, kde_train_support)
        mdfpc_fc = forecaster.predict(horizon=HORIZON, var_lags=1, forecast_index=test_date)

        Y_hat_t = mdfpc_fc["future_L2_curves"]
        lambda_inv_Y_hat_t_supp = mdfpc_fc["future_supports"]
        lambda_inv_Y_hat_t = mdfpc_fc["future_densities"]

        # T(f_kde forecast) x T(f_kde)
        append_curve_comparison(
            measures,
            scenario=id_scenario_kde,
            n_rep=n_rep,
            model_name=model_name,
            fold=fold,
            comparison="yHatFc_Y",
            fc_date=fc_date,
            test=Y_kde_t,
            forecast=Y_hat_t
        )

        # T(f_kde forecast) x T(f_conditional)
        append_curve_comparison(
            measures,
            scenario=id_scenario_kde,
            n_rep=n_rep,
            model_name=model_name,
            fold=fold,
            comparison="yHatFc_Tf_conditional",
            fc_date=fc_date,
            test=Y_cond_t,
            forecast=Y_hat_t
        )

        # # Lambda^{-1}(T(f_kde forecast)) x f_kde
        # df_supp, df_kde, df_fc = fdaUtils.align_densities(
        #     f_hat_t_supp,
        #     f_hat_t,
        #     lambda_inv_Y_hat_t_supp,
        #     lambda_inv_Y_hat_t,
        #     lambda_inv_Y_hat_t.columns
        # )
        # append_curve_comparison(
        #     measures,
        #     scenario=id_scenario_kde,
        #     n_rep=n_rep,
        #     model_name=model_name,
        #     fold=fold,
        #     comparison="fHatFc_fHat",
        #     fc_date=fc_date,
        #     test=df_kde,
        #     forecast=df_fc,
        #     support=df_supp.iloc[:, 0].values
        # )

        # Lambda^{-1}(T(f_kde forecast)) x f_conditional
        # df_supp, df_cond, df_fc = fdaUtils.align_densities(
        #     f_t_support,
        #     f_t,
        #     lambda_inv_Y_hat_t_supp,
        #     lambda_inv_Y_hat_t,
        #     lambda_inv_Y_hat_t.columns
        # )
        # append_curve_comparison(
        #     measures,
        #     scenario=id_scenario_kde,
        #     n_rep=n_rep,
        #     model_name=model_name,
        #     fold=fold,
        #     comparison="fHatFc_f",
        #     fc_date=fc_date,
        #     test=df_cond,
        #     forecast=df_fc,
        #     support=df_supp.iloc[:, 0].values
        # )

    joblib.dump(measures, f"{CV_FCS_PATH}cvFc___scenario_{id_scenario_kde}___rep_{n_rep}___kde_{model_name_path}.jbl")

    del scenario_kde_dict, scenario_sim_dict, df_support, df_densities, sim_densities, sim_densities_supp
    del scenario_sim_lqd, scenario_mlqdt, scenario_kde_lqd, measures
    gc.collect()
    pbar.update(1)

Total CV Progress:   0%|          | 0/80 [00:00<?, ?it/s]

{'scenario': 'scenario_1___nu=3', 'n_rep': 0, 'model_name': 'gaussian_rot', 'address': '../data/interim/simulation/20260604_02/02_kdes/scenario_1___nu=3___rep_0___kde_gaussian_rot.jbl'}
		>>> cv 1/1


ValueError: p and q must have the same length

# Results

In [ ]:
cv_results = []
cv_files = [str(Path(CV_FCS_PATH) / x) for x in os.listdir(CV_FCS_PATH) if x.endswith(".jbl")]
for file in cv_files:
    cv_results.append(pd.DataFrame(joblib.load(file)))
df_cv_results = pd.concat(cv_results)
df_cv_results.sort_values(by=["scenario", "n_rep", "model_name"], inplace=True)
df_cv_results.reset_index(inplace=True, drop=True)

In [ ]:
df_cv_results.comparison.unique()

In [ ]:
df_cv_results.groupby("model_name")["KLD"].mean().sort_values()

In [ ]:
# Extract the first two parts only
split_data = df_cv_results['scenario'].str.split("___", expand=True)
df_cv_results["scenario_id"] = split_data[0]
df_cv_results["t_df"] = split_data[1]

df_cv_results.groupby(["scenario_id", "t_df", "comparison", "model_name"], as_index=False)[["KLD", "L_1", "L_2", "L_INFTY"]].mean().sort_values(by=["scenario_id", "comparison", "t_df", "KLD"])

In [ ]:



df_plot = df_cv_results.copy()

# Clean labels
df_plot["model_name"] = (
    df_plot["model_name"]
    .str.replace(" rot", "", regex=False)
    .str.replace("t-student", "t", regex=False)
)

# Better style
sns.set_theme(
    style="whitegrid",
    context="paper",  
    font_scale=1.2
)

g = sns.catplot(
    data=df_plot,
    kind="box",
    x="model_name",
    y="KLD",
    col="comparison",
    row="t_df",
    sharey=False,
    height=3.8,
    aspect=1.3,
    linewidth=1,
    fliersize=0,   
    palette="Set2"
)

# Overlay points (controlled)
for ax in g.axes.flat:
    sns.stripplot(
        data=df_plot,
        x="model_name",
        y="KLD",
        ax=ax,
        color="black",
        size=1.8,
        alpha=0.25,
        jitter=0.25
    )

# Log scale (CRUCIAL here)
for ax in g.axes.flat:
    ax.set_yscale("log")

# Titles (cleaner)
g.set_titles(
    row_template="df = {row_name}",
    col_template="{col_name}"
)

g.set_axis_labels("", "KLD (log scale)")

# Rotate nicely
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

Verificar: Comparar $Y$ com $Y_{d=3}$; problema: curvas distantes

In [ ]:
for scenario in df_cv_results.scenario.unique():
    df_cv_2 = df_cv_results[df_cv_results["scenario"]==scenario]
    for kde_model_name in df_cv_2.model_name.unique():
        df_cv_3 = df_cv_2[df_cv_2["model_name"]==kde_model_name]
        for target in df_cv_3.comparison.unique():
            if "_Y" in target or "Xboot" in target:
                ylim = [-10, 10]
                xlim = [-0.05, 1.05]
            else:
                ylim = [-0.5, 1.1]
                xlim = [-10, 10]
            df_cv_4 = df_cv_3[df_cv_3.comparison == target]
            plt.figure(figsize=(15,5))
            for n, x in df_cv_4.iterrows():
                curves = x["curves"].reset_index()[["support", "actual", "forecast"]].set_index("support")
                # plt.plot(curves["actual"], color="gray", alpha=0.1)
                # plt.plot(curves["forecast"], color="blue", alpha=0.1)
                curves["actual"].plot(color="gray", alpha=0.1)
                curves["forecast"].plot(color="blue", alpha=0.1)
            plt.title(f"Forecasts | base pdf: {scenario} | {kde_model_name} | {target}")
            plt.ylim(ylim)
            plt.xlim(xlim)
            plt.show()

            plt.figure(figsize=(15,5))
            for n, x in df_cv_4.iterrows():
                curves = x["curves"].reset_index()[["support", "actual", "forecast"]].set_index("support")
                curves["residuals"] = curves.actual - curves.forecast
                plt.plot(curves["residuals"], color="gray", alpha=0.1)
            plt.title(f"Residuals | base pdf: {scenario} | {kde_model_name} | {target}")
            plt.ylim(ylim)
            plt.xlim(xlim)
            plt.show()

            plt.figure(figsize=(15,5))
            for n, x in df_cv_4.iterrows():
                curves = x["curves"].reset_index()[["support", "actual", "forecast"]].set_index("support")
                curves["residuals"] = curves.actual - curves.forecast
                curves["rel_resid"] = (np.abs(curves["residuals"])/curves.actual).fillna(0)
                plt.plot(curves["rel_resid"], color="gray", alpha=0.1)
            plt.title(f"Relative residuals | base pdf: {scenario} | {kde_model_name} | {target}")
            plt.ylim(ylim)
            plt.xlim(xlim)
            plt.show()